# 03 - Segmentation et morphologie

**Objectif de l'etape.** Isoler la voiture dans une ROI initiale manuelle, comparer Otsu et le seuillage adaptatif, puis nettoyer le masque par morphologie.

**Methode.** Otsu choisit un seuil global dans la ROI. Le seuillage adaptatif utilise des seuils locaux. L'ouverture supprime les petits bruits, la fermeture remplit les petits trous et la plus grande composante connectee conserve l'objet principal.

**Lien avec le sujet.** Un masque plus propre stabilise les points Lucas-Kanade suivis sur la voiture.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATASET_DIR = PROJECT_ROOT / 'data' / 'car' / 'car-11' / 'img'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

image_files = sorted([p for p in DATASET_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
print('Nombre de frames:', len(image_files))

Nombre de frames: 1661


In [2]:
# Bbox manuelle de depart: ajuster si necessaire ou utiliser cv2.selectROI dans l'interface Tkinter.
# Format: x, y, w, h. Cette valeur ne vient pas d'annotations.
initial_bbox = (535, 300, 220, 105)
print('initial_bbox =', initial_bbox)

initial_bbox = (535, 300, 220, 105)


In [3]:
import cv2
from src.preprocessing import preprocess_image_with_method
from src.detection import segment_roi, clean_segmentation_mask, largest_connected_component, bbox_from_mask, adaptive_threshold, otsu_threshold
from src.visualization import draw_bbox, draw_mask_overlay, save_comparison_grid, save_frame

frame = cv2.imread(str(image_files[0]))
gray = preprocess_image_with_method(frame, method='stretching')['enhanced']
x, y, w, h = initial_bbox
roi = gray[y:y+h, x:x+w]
otsu = otsu_threshold(roi, invert=True)
adaptive = adaptive_threshold(roi, invert=True)
clean = clean_segmentation_mask(otsu, kernel_size=(5, 5))
largest = largest_connected_component(clean)
segmented_bbox = bbox_from_mask(largest, parent_bbox=initial_bbox)
print('bbox estimee depuis masque:', segmented_bbox)

save_frame(otsu, RESULTS_DIR / 'segmentation' / 'notebook_otsu_mask.png')
save_frame(adaptive, RESULTS_DIR / 'segmentation' / 'notebook_adaptive_mask.png')
save_frame(clean, RESULTS_DIR / 'morphology' / 'notebook_clean_mask.png')
save_frame(draw_mask_overlay(frame, largest, bbox=initial_bbox), RESULTS_DIR / 'morphology' / 'notebook_mask_overlay.png')
save_comparison_grid({'roi': roi, 'otsu': otsu, 'adaptive': adaptive, 'clean': clean, 'largest': largest, 'bbox': draw_bbox(frame, segmented_bbox)}, RESULTS_DIR / 'final_visualization' / 'notebook_segmentation_morphology.png')

bbox estimee depuis masque: (542, 314, 198, 91)


True

**Resultats generes.** Masque Otsu, masque adaptatif, masque nettoye, plus grande composante et overlay sont sauvegardes dans `results/segmentation/`, `results/morphology/` et `results/final_visualization/`.

**Interpretation.** La segmentation dans la ROI evite de seuiller toute la scene. Si Otsu isole mieux la voiture, il est garde pour la suite. Les erreurs possibles viennent de la route, des ombres, du faible contraste et de morceaux d'arriere-plan inclus dans la ROI.